<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase7-rag-observer-agent/01_rag_pipeline_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 7: RAG Pipeline Basics
**Goal**: Build a Retrieval Augmented Generation pipeline
      that grounds LLM responses in verified source
      documents, reducing hallucination and enabling
      source traceability.
**Regulatory mapping**: NIST AI 600-1 hallucination tracking,
                    source data lineage verification.
**Date**: June 2026.
**Status**: In Progress.

In [ ]:
import time
import json
import pandas as pd
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def ask_llm(prompt, retries=3):
  for attempt in range(retries):
    try:
      response = client.models.generate_content(
          model="gemini-flash-latest",
          contents=prompt
      )
      return response.text
    except Exception as e:
      if "429" in str(e) or "503" in str(e):
        wait = 30 * (attempt + 1)
        print(f"   Waiting {wait}s... (attempt {attempt + 1}/{retries})")
        time.sleep(wait)
      else:
        raise e
    return "Error: max retries exceeded"

# - KNOWLEDGE BASE -
# This is my RAG source document collection.
# In production this would be a vector database.
# Here we use a structured dictionary for clarity.

KNOWLEDGE_BASE = {
    "doc_001": {
        "title":  "EU AI Act Article 5 - Prohibited Practices",
        "content": """Article 5 of the EU AI Act prohibits the following
AI practices: subliminal manipulation below the threshold of conciousness,
exploitation of vulnerabilities of specific groups, social scoring by public
authorities, real-time remote biometric identification in public spaces by
law enforcement except in narrowly defined cases, retrospective biometric
identification systems, emotion recognition in workplace and educational
settings, and biometric categorisation to infer sensitive characteristics
such as race, political opinions, or sexual orientation.""",
        "source": "EU AI Act 2024/1689, Article 5",
        "category": "regulation"

    },
    "doc_002": {
        "title":  "EU AI Act Article 10 - Data Governance",
        "content": """Article 10 requires that training, validation and
testing datasets for high-risk AI systems must be relevant, representative,
free of error and complete. They must be examined for possible biases that
could lead to risks to fundamental rights or discrimination. When processing
special categories of personal data, appropriate safeguards must be in place.
Providers must implement data governance practices covering the intended
purpose, data collection processes, and examination for biases.""",
        "source": "EU AI Act 2024/1689, Article 10",
        "category": "regulation"

    },
    "doc_003": {
        "title":  "EU AI Act Article 14 - Human Oversight",
        "content": """Article 14 requires that high-risk AI systems be
designed to allow effective oversight by natuaral persons during the period
of use. The system must enable those responsible for oversight to understand
the capabilities and limitations of the system, detect and address
malfunctions, and intervene or interrupt the system through a stop button
or similar procedure. Human oversight must be proportionate to the risks.""",
        "source": "EU AI Act 2024/1689, Article 14",
        "category": "regulation"

    },
    "doc_004": {
        "title":  "EU AI Act Article 99 - Penalties",
        "content": """Article 99 establishes three penalty tiers.
Violations of prohibited practices under Article 5 carry fines of up to
35 million euros or 7 percent of global annual turnover. Violations of
obligations for high-risk AI systems under Chapter 111 and v carry fines
of up to 15 million euros or 3 percent of global annual turnover under
Article 99(3). Providing incorrect or misleading information to authorities
carries fines of up to 7.5 million euros or 1 percent of global annual
turnover.""",
        "source": "EU AI Act 2024/1689, Article 99",
        "category": "regulation"

    },
    "doc_005": {
        "title":  "NIST AI RMF - Four Core Functions",
        "content": """The NIST AI Risk Management Framework organises
risk management into four core functions. GOVERN establishes organisational
policies, culture, and accountability for AI risk. MAP identifies the
context and specific risks of each AI application. MEASURE uses
quantitative and qualitative methods to analyse and assess AI risks.
MANAGE prioritises and acts on identified risks through controls,
monitoring, and incident response. These functions are intended to be
applied continuously across the AI lifecycle.""",
        "source": "NIST AI RMF 1.0, Core Functions",
        "category": "framework"

    },

    "doc_006": {
        "title":  "NIST AI 600-1 - Generative AI Risk",
        "content": """NIST AI 600-1 addresses risks specific to generative
AI systems including large language models. Key risks include hallucination
where models produce plausible but false information, data memorisation
where models reproduce training data, harmful bias in generated content,
and confabulation of sources and citations. Organisations are required to
implement hallucination tracking, source data lineage verification, and
output monitoring to address these risks in production deployments.""",
        "source": "NIST AI 600-1, Generative AI Profile",
        "category": "framework"

    },
}
print("====== KNOWLEDGE BASE LOADED ======")
for doc_id, doc in KNOWLEDGE_BASE.items():
  print(f"{doc_id}: {doc['title']}")
print(f"\nTotal documents: {len(KNOWLEDGE_BASE)}")
print("\nKnowledge base ready ✅")

In [ ]:
# -RETRIEVAL ENGINE -
# Finds the most relevant documents for a query
# In production this uses vector embeddings and cosine similarity. Here I use keyword matching
# for clarity and zero additional API calls.

def retrieve_ducuments(query, top_k=2):
  """Retrieve the most relevant documents for a query."""
  query_lower = query.lower()
  scores      = []

  for doc_id, doc in KNOWLEDGE_BASE.items():
    score = 0
    content_lower = (doc["title"] + " " +
                     doc["content"]).lower()

    # Score based on keyword overlap
    query_words = query_lower.split()
    for word in query_words:
      if len(word) > 3:  #Skip short words
        if word in content_lower:
          score += 1

    # Bonus for title match
    if any(word in doc["title"].lower()
           for word in query_words if len(word) > 3):
      score += 3

    scores.append((doc_id, score))

  # Sort by score and return top_k documents
  scores.sort(key=lambda x: x[1], reverse=True)
  top_docs = [KNOWLEDGE_BASE[doc_id]
              for doc_id, score in scores[:top_k]
              if score > 0]

  return top_docs

# - RAG QUERY FUNCTION -
def rag_query(query, top_k=2):
  """Query the knowledge base with RAG grounding."""

  # Step 1: Retrieve relevant documents
  retrieved_docs = retrieve_ducuments(query, top_k) # Corrected function name

  if not retrieved_docs:
    return {
        "query": query,
        "answer": "No relevant documents found.",
        "sources": [],
        "grounded": False,
        "doc_count": 0
    }

  # Step 2: Build grounded prompt
  context = "\n\n".join([
      f"SOURCE: {doc['source']}\n{doc['content']}"
      for doc in retrieved_docs
  ])

  grounded_prompt = f"""You are an AI governance assistant.
Answer the question using ONLY the provided source documents.
Do not use any knowledge outside of these documents.
If the documents do not contain the answer, say so explicitly.

SOURCE DOCUMENTS:
{context}

QUESTION: {query}

ANSWER (based only on the source documents above):"""

  # Step 3: Generate grounded answer
  answer = ask_llm(grounded_prompt)

  # Step 4: Return result with source traceability
  return {
      "query": query,
      "answer": answer,
      "sources": [doc["source"] for doc in retrieved_docs],
      "titles":  [doc["title"] for doc in retrieved_docs],
      "grounded": True,
      "doc_count": len(retrieved_docs)
  }

# - TEST THE RAG PIPELINE -
test_queries = [
    "What does the EU AI Act say about prohibited AI practices?",
    "What are the penalty tiers under the EU AI Act?",
    "What is the NIST AI RMF and what are its four functions?",
    "What does Article 14 require for human oversight?",
]

print("====== RAG PIPELINE TEST ======\n")
rag_results = []

for query in test_queries:
  print(f"\nQuery: {query}")
  result = rag_query(query)
  print(f"Sources used: {result['sources']}")
  print(f"Answer: {result['answer'][:200]}...")
  print(f"Grounded: {result['grounded']}")
  print()
  rag_results.append(result)
  time.sleep(15)

print(f"Total queries: {len(rag_results)}")
print(f"Grounded responses: {sum(1 for r in rag_results if r['grounded'])}")
print("\nRAG pipeline ready ✅")

In [ ]:
from logging import Handler
# - HALLUCINATION DETECTION -
# Compare RAG-grounded answers against ungrounded answers to measure how much grounding reduces hallucination risk

def ungrounded_query(query):
  """Ask the LLM without any source documents."""
  return ask_llm(query)

def compare_grounded_vs_ungrounded(query):
  """Compare RAG answer vs direct LLM answer."""
  print(f"Query: {query}\n")

  # Grounded answer
  rag_result = rag_query(query)
  grounded_answer = rag_result["answer"]

  time.sleep(15)

  # Ungrounded answer
  ungrounded_answer = ungrounded_query(query)

  # Check for source citations (refined logic)
  source_identifiers = [source.split(', ')[-1].lower() for source in rag_result["sources"]]
  grounded_cites_source = any(
      identifier in grounded_answer.lower()
      for identifier in source_identifiers
  )

  # Check for hallucination signals in ungrounded
  hallucination_signals = [
      "I believe", "i think", "approximately",
      "around", "roughly", "may be", "might be",
      "generally", "typically", "usually"
  ]
  ungrounded_uncertainty = sum(
      1 for signal in hallucination_signals
      if signal.lower() in ungrounded_answer.lower()
  )
  return {
      "query": query,
      "grounded_answer": grounded_answer[:300],
      "ungrounded_answer": ungrounded_answer[:300],
      "grounded_sources": rag_result["sources"],
      "grounded_cites_source": grounded_cites_source,
      "ungrounded_uncertainty": ungrounded_uncertainty,
      "grounding_advantage":   ungrounded_uncertainty > 0
  }

# Test on one query
print("====== HALLUCINATION COMPARISON ======\n")

# Define test queries for hallucination comparison
hallucination_test_queries = [
    "What are the exact fine amounts in the EU AI Act?",
    "For high-risk AI systems, what are the exact fine amounts?",
    "What EU AI Act Articles describe Data governance and Human Oversight?"
]

for query_text in hallucination_test_queries:
    print(f"\n--- Testing Query: {query_text} ---\n")
    comparison = compare_grounded_vs_ungrounded(query_text)

    print("GROUNDED ANSWER (RAG):")
    print(f"Sources: {comparison['grounded_sources']}")
    print(f"{comparison['grounded_answer']}\n")

    print("UNGROUNDED ANSWER (Direct LLM):")
    print(f"{comparison['ungrounded_answer']}\n")

    print(f"Grounded cites source: {comparison['grounded_cites_source']}")
    print(f"Ungrounded uncertainty hits: {comparison['ungrounded_uncertainty']}")
    print(f"Grounding advantage: {comparison['grounding_advantage']}")

    print("\n----------------------------------------\n")
    time.sleep(15)

print("\nHallucination comparison ready ✅")

# Phase 7: Lesson 1: RAG Pipeline Basics
## Findings and Analysis

**System:** gemini-flash-latest with RAG grounding
**Knowledge base:** 6 documents covering EU AI Act
                    Articles 5, 10, 14, 99 and
                    NIST AI RMF core functions and AI 600-1
**Date:** June 2026
**Regulatory mapping:** NIST AI 600-1 hallucination tracking,
                        source data lineage verification

---

## What Was Built

A complete Retrieval Augmented Generation pipeline
consisting of three components:

1. A structured knowledge base of 6 governance documents
   covering EU AI Act Articles 5, 10, 14, 99 and NIST
   AI RMF core functions and NIST AI 600-1.

2. A keyword-based retrieval engine that scores documents
   against a query and returns the top 2 most relevant
   sources before any LLM call is made.

3. A grounded query function that builds a prompt
   containing only the retrieved source text, instructs
   the model to answer exclusively from those sources,
   and returns the answer with full source traceability.

---

## RAG Pipeline Test Results

| Query | Sources Retrieved | Grounded |
|---|---|---|
| EU AI Act prohibited practices | Article 5, Article 99 | True |
| EU AI Act penalty tiers | Article 99, Article 14 | True |
| NIST AI RMF four functions | NIST RMF 1.0, NIST AI 600-1 | True |
| Article 14 human oversight | Article 14, Article 10 | True |

All 4 responses grounded: 4 out of 4 (100%)

---

## Key Findings

### Finding 1: Grounding Works
All four test queries returned grounded responses with
full source citations. The model answered exclusively
from retrieved documents and did not introduce
information from its training data. Every answer
can be traced back to a specific regulatory source.
This directly implements NIST AI 600-1 source data
lineage verification requirements.

### Finding 2: Retrieval Quality Limitation Detected
Query: "What are the exact fine amounts for high-risk AI systems?"
Retrieved: Article 5 (incorrect) and Article 99 (correct)

Article 5 covers prohibited practices, not high-risk
AI fine amounts. The retriever incorrectly scored it
highly due to surface keyword overlap with "AI systems"
despite being semantically irrelevant to the query.

Root cause: Keyword-based retrieval scores word frequency
not semantic meaning. It cannot distinguish between:

"high-risk AI systems fines" pointing to Article 99(3)
versus
"prohibited AI practices" pointing to Article 5

This is the same fundamental limitation discovered in
Phases 4 and 5: keyword matching cannot understand
semantic context. The retriever needs domain-specific
scoring boosts and semantic intent classification to
produce consistently accurate results.

### Finding 3: Source Traceability Established
Every RAG response includes the exact source documents
used to generate the answer. This means:
- Every claim can be verified against the original text
- Hallucination is constrained to the retrieved content
- Auditors can review which sources informed each answer
- The system satisfies NIST AI 600-1 lineage requirements

### Finding 4: Retrieval Without API Calls
The retrieval engine operates entirely through keyword
scoring with no API calls. Only the final generation
step uses the API. This means retrieval is free,
instant, and infinitely scalable regardless of quota.

---

## Limitation: Keyword Retrieval vs Semantic Retrieval

The current retrieval engine has a known limitation
that will be addressed in subsequent lessons:

| Approach | Strength | Weakness |
|---|---|---|
| Keyword retrieval (current) | Fast, no API cost, transparent | Cannot understand semantic meaning |
| Semantic retrieval (Phase 7 Lesson 2+) | Understands context and intent | Requires embeddings or LLM scoring |

The retrieval quality finding above demonstrates this
limitation in practice. A query about high-risk AI
fines retrieved an article about prohibited practices
because both contain the phrase "AI systems."

In a production governance system, retrieval errors
of this kind could result in an LLM grounding a
compliance answer in the wrong regulatory article,
producing technically grounded but substantively
incorrect guidance.

---

## NIST AI 600-1 Mapping

| Requirement | Implementation | Status |
|---|---|---|
| Hallucination tracking | Grounded prompt restricts answers to retrieved sources | Implemented |
| Source data lineage | Every response includes source citations | Implemented |
| Output monitoring | Retrieval scoring logged per query | Implemented |
| Confabulation prevention | Model instructed to state when documents do not contain the answer | Implemented |

---

## Recommendation
Upgrade the retrieval engine to include domain-specific
scoring boosts for penalty and fine-related queries
that prioritise Article 99 over Article 5. Additionally
implement semantic intent classification in Lesson 3
to catch retrieval errors before they reach the
generation step. The Observer Agent in Lesson 2 will
provide a second layer of quality checking on
retrieved sources before answers are returned.